# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BilaalBakare/Flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method: Logistic Regression

My target is whether a page's average position stays poor (21+) or CTR falls below the bucket average, predicted from earlier-month signals (position, impressions, click history) — a direct extension of my Week 4 baseline's binary flag logic, so both can be judged on the same target.

I'm choosing Logistic Regression because: (1) it directly fits a binary classification problem, (2) its coefficients are interpretable — I can see which signals actually drive the prediction, continuing the "why" focus from my Week 1 signal checks, and (3) it's a natural, proportionate upgrade from my baseline's simple threshold rule, rather than jumping to an unnecessarily complex model the brief specifically warns against rewarding for its own sake.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from dotenv import load_dotenv
import os

load_dotenv()
hf_token = os.getenv("HF_TOKEN")


In [2]:
import duckdb

con = duckdb.connect()

con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: Grouped by page (content_hash_id)

I'm using a grouped split by page, not a random row split, because my data's grain is page × day — the same handful of pages repeat across many dates (confirmed in my Week 4 top-20 review, where 8 of 20 rows were the same single page). A random split would let the same page appear in both training and test sets on different dates, letting the model partially memorize that page rather than genuinely generalize — a form of leakage similar to the label-leakage risk from Week 2. Grouping by content_hash_id ensures every row for a given page falls entirely on one side of the split, so my test score reflects performance on pages the model has never seen.

In [3]:
feature_df = con.sql(f"""
    WITH base AS (
        SELECT
            client_hash_id,
            content_hash_id,
            report_date,
            gsc_avg_position,
            gsc_impressions,
            gsc_clicks,
            gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0) AS ctr
        FROM read_parquet('{rel}/fact_content_daily_performance/**/*.parquet')
        WHERE month = '2026-03'
          AND gsc_avg_position IS NOT NULL
          AND gsc_impressions > 0
    ),
    bucket_avg AS (
        SELECT AVG(ctr) AS overall_avg_ctr FROM base
    )
    SELECT
        base.*,
        CASE WHEN base.ctr < bucket_avg.overall_avg_ctr THEN 1 ELSE 0 END AS y_low_ctr
    FROM base, bucket_avg
""").df()

feature_df.shape

(3611061, 8)

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.model_selection import GroupShuffleSplit

# Build features, target, and groups from feature_df
X = feature_df[['gsc_avg_position', 'gsc_impressions']]
y = feature_df['y_low_ctr']
groups = feature_df['content_hash_id']

print(y.value_counts())

# Grouped split so no page appears in both train and test
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

train_df = feature_df.iloc[train_idx]
test_df = feature_df.iloc[test_idx]

# Sanity check: confirm no page appears in both sets
overlap = set(train_df['content_hash_id']) & set(test_df['content_hash_id'])
print("Pages in both train and test:", len(overlap))  # should be 0

y_low_ctr
1    3250616
0     360445
Name: count, dtype: int64
Pages in both train and test: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report

# --- Train the model ---
X_train, y_train = train_df[['gsc_avg_position', 'gsc_impressions']], train_df['y_low_ctr']
X_test, y_test = test_df[['gsc_avg_position', 'gsc_impressions']], test_df['y_low_ctr']

model = LogisticRegression(class_weight='balanced')
model.fit(X_train, y_train)

y_pred_model = model.predict(X_test)

# --- Recreate baseline's rule as a prediction on the same test set ---
y_pred_baseline = (
    (test_df['gsc_avg_position'] >= 21) &
    (test_df['gsc_impressions'] >= 50)
).astype(int)

# --- Compare on the same metric ---
results = {
    'Baseline (Week 4 rule)': {
        'precision': precision_score(y_test, y_pred_baseline),
        'recall': recall_score(y_test, y_pred_baseline),
        'f1': f1_score(y_test, y_pred_baseline),
    },
    'Logistic Regression (Week 5)': {
        'precision': precision_score(y_test, y_pred_model),
        'recall': recall_score(y_test, y_pred_model),
        'f1': f1_score(y_test, y_pred_model),
    }
}

import pandas as pd
comparison_df = pd.DataFrame(results).T
comparison_df

,precision,recall,f1
Baseline (Week 4 rule),0.857519,0.048466,0.091747
Logistic Regression (Week 5),0.957154,0.846517,0.898442


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.